# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eminamandzukic/FlyRank_starter/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The playbook turns the validated output into a ranked queue for human review, not automatic content changes.

Actions:
- `review_for_refresh` - content with meaningful search visibility and an average search position worse than 10.
- `protect_and_monitor` - visible content currently ranking within the top 10.

Reason codes:
- `visible_but_not_top10`
- `strong_current_position`

The final ranking follows the rule-based baseline because it performed better than the learned models at the top of the validation queue. These actions are decision-support signals rather than causal explanations.

In [15]:
import getpass
import duckdb
import calendar
import pandas as pd
import numpy as np

from huggingface_hub import hf_hub_download

HF_TOKEN = getpass.getpass(
    "Hugging Face READ token: "
).strip()

con = duckdb.connect()

clients_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="dim_clients.parquet",
    token=HF_TOKEN
)

In [16]:
def build_month_pair(feature_month, outcome_month):
    feature_path = hf_hub_download(
        repo_id="FlyRank/internship-warehouse",
        repo_type="dataset",
        filename=f"fact_content_daily_performance/month={feature_month}/data_0.parquet",
        token=HF_TOKEN
    )

    outcome_path = hf_hub_download(
        repo_id="FlyRank/internship-warehouse",
        repo_type="dataset",
        filename=f"fact_content_daily_performance/month={outcome_month}/data_0.parquet",
        token=HF_TOKEN
    )

    feature_year, feature_num = map(int, feature_month.split("-"))
    outcome_year, outcome_num = map(int, outcome_month.split("-"))

    feature_days = calendar.monthrange(feature_year, feature_num)[1]
    outcome_days = calendar.monthrange(outcome_year, outcome_num)[1]

    feature_start = f"{feature_month}-01"

    frame = con.sql(f"""
        WITH features AS (
            SELECT
                client_hash_id,
                content_hash_id,
                SUM(gsc_impressions) AS impressions_feature,
                SUM(gsc_clicks) AS clicks_feature,
                AVG(gsc_avg_position) AS avg_position_feature
            FROM read_parquet('{feature_path}')
            GROUP BY client_hash_id, content_hash_id
        ),
        outcomes AS (
            SELECT
                client_hash_id,
                content_hash_id,
                SUM(gsc_impressions) AS impressions_outcome
            FROM read_parquet('{outcome_path}')
            GROUP BY client_hash_id, content_hash_id
        )
        SELECT
            f.client_hash_id,
            f.content_hash_id,
            f.impressions_feature,
            f.clicks_feature,
            f.avg_position_feature,
            o.impressions_outcome
        FROM features f
        INNER JOIN outcomes o
            ON f.client_hash_id = o.client_hash_id
           AND f.content_hash_id = o.content_hash_id
        INNER JOIN read_parquet('{clients_path}') c
            ON f.client_hash_id = c.client_hash_id
        WHERE
            f.impressions_feature >= 200
            AND c.gsc_data_start <= DATE '{feature_start}'
    """).df()

    frame["ctr_feature"] = (
        frame["clicks_feature"] /
        frame["impressions_feature"]
    )

    frame["avg_daily_impressions_feature"] = (
        frame["impressions_feature"] / feature_days
    )

    frame["avg_daily_impressions_outcome"] = (
        frame["impressions_outcome"] / outcome_days
    )

    frame["is_declining"] = (
        frame["avg_daily_impressions_outcome"]
        < 0.8 * frame["avg_daily_impressions_feature"]
    ).astype(int)

    frame["feature_month"] = feature_month
    frame["outcome_month"] = outcome_month

    return frame

In [17]:
training_pairs = [
    ("2025-02", "2025-03"),
    ("2025-03", "2025-04"),
    ("2025-04", "2025-05"),
    ("2025-05", "2025-06"),
    ("2025-06", "2025-07"),
    ("2025-07", "2025-08"),
    ("2025-08", "2025-09"),
    ("2025-09", "2025-10"),
    ("2025-10", "2025-11"),
    ("2025-11", "2025-12"),
    ("2025-12", "2026-01"),
    ("2026-01", "2026-02"),
    ("2026-02", "2026-03"),
    ("2026-03", "2026-04"),
]

training_frames = [
    build_month_pair(feature_month, outcome_month)
    for feature_month, outcome_month in training_pairs
]

train_frame = pd.concat(
    training_frames,
    ignore_index=True
)

april_may = build_month_pair(
    "2026-04",
    "2026-05"
)

print("Training rows:", len(train_frame))
print("Queue rows:", len(april_may))

Training rows: 358498
Queue rows: 86282


In [18]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import numpy as np

feature_cols = [
    "impressions_feature",
    "clicks_feature",
    "avg_position_feature",
    "ctr_feature",
]

X_train = train_frame[feature_cols]
y_train = train_frame["is_declining"]

X_queue = april_may[feature_cols]

model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

model.fit(X_train, y_train)

queue = april_may.copy()
queue["decline_risk"] = model.predict_proba(X_queue)[:, 1]

In [19]:
queue["baseline_score"] = np.where(
    queue["avg_position_feature"] > 10,
    queue["impressions_feature"],
    0
)

queue["action"] = np.where(
    queue["baseline_score"] > 0,
    "review_for_refresh",
    "protect_and_monitor"
)

queue["reason_code"] = np.where(
    queue["baseline_score"] > 0,
    "visible_but_not_top10",
    "strong_current_position"
)

queue = queue.sort_values(
    ["baseline_score", "impressions_feature"],
    ascending=[False, False]
).reset_index(drop=True)

queue["rank"] = np.arange(1, len(queue) + 1)

queue[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "baseline_score",
        "impressions_feature",
        "avg_position_feature",
        "ctr_feature",
        "action",
        "reason_code",
    ]
].head(10)

,rank,client_hash_id,content_hash_id,baseline_score,impressions_feature,avg_position_feature,ctr_feature,action,reason_code
0,1,client_cd12bcfd98942aa1,content_99fc6465edb0e52c,267960.0,267960.0,10.408285,0.000004,review_for_refresh,visible_but_not_top10
1,2,client_23a62021009f63c4,content_661a7734f691bef5,180523.0,180523.0,22.968435,0.000648,review_for_refresh,visible_but_not_top10
2,3,client_23a62021009f63c4,content_e8a52cf3d5988c07,173699.0,173699.0,11.715300,0.003523,review_for_refresh,visible_but_not_top10
3,4,client_23a62021009f63c4,content_36e53e9c707674fc,151400.0,151400.0,28.866542,0.002107,review_for_refresh,visible_but_not_top10
4,5,client_23a62021009f63c4,content_df47d1b976106de4,118187.0,118187.0,21.601828,0.000956,review_for_refresh,visible_but_not_top10
5,6,client_3f0ce4d44fe94f3d,content_468d0aa0891d425d,114336.0,114336.0,12.760730,0.001618,review_for_refresh,visible_but_not_top10
6,7,client_23a62021009f63c4,content_93a9b8328d4fd032,113829.0,113829.0,22.833910,0.000589,review_for_refresh,visible_but_not_top10
7,8,client_23a62021009f63c4,content_14df7b049d1d6467,113233.0,113233.0,26.832023,0.000583,review_for_refresh,visible_but_not_top10
8,9,client_20259bd6705d81d4,content_82e35c4845e6c391,109367.0,109367.0,27.999795,0.000174,review_for_refresh,visible_but_not_top10
9,10,client_e5c2aa26a8598242,content_39457d17e716086c,101721.0,101721.0,25.434874,0.000324,review_for_refresh,visible_but_not_top10


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

This playbook is intended for human-reviewed content prioritization. It helps identify which visible content items may deserve review first based on measured historical search signals.

It should not be used to automatically edit, publish, remove, or de-prioritize content.

The priority ranking is directional rather than certain. Week-6 validation showed weaker performance under the time-aware split, so recommendations should be combined with editorial judgment and page context.

The playbook is most appropriate for content with enough historical search visibility to support a meaningful signal.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Every recommended action requires human review before any content change is made.

Reviewer checks should include:
- whether the page is strategically important,
- whether the decline signal matches the page context,
- whether the content is outdated or incomplete,
- and whether another explanation could account for the measured change.

The system should NOT automatically:
- rewrite or publish content,
- delete or prune pages,
- change titles or metadata,
- infer Google ranking causes,
- or treat a high priority ranking as proof that a refresh will improve performance.

The queue is for prioritization only. Final actions remain human decisions.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

The playbook should be monitored over time rather than treated as permanently valid.

Useful triggers for review or retraining include:
- a noticeable change in decline base rate,
- weaker Precision@K on newer months,
- changes in the relationship between position, impressions, CTR, and decline,
- or major shifts in the content portfolio.

A simple check can be run monthly. Retraining is only needed if measured performance or data patterns drift enough to reduce decision usefulness.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [20]:
from pathlib import Path

outputs_dir = Path("../outputs")
outputs_dir.mkdir(parents=True, exist_ok=True)

queue_export = queue[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "baseline_score",
        "decline_risk",
        "impressions_feature",
        "clicks_feature",
        "avg_position_feature",
        "ctr_feature",
        "action",
        "reason_code",
    ]
].copy()

queue_path = outputs_dir / "w07_ranked_action_queue.csv"
queue_export.to_csv(queue_path, index=False)

print("Saved:", queue_path)
print("Rows:", len(queue_export))

Saved: ../outputs/w07_ranked_action_queue.csv
Rows: 86282


In [21]:
import json

metrics = {
    "validation_design": "time-aware",
    "validation_period": "2026-04 to 2026-05",
    "base_rate": 0.576,

    "baseline_roc_auc": 0.554,
    "baseline_precision_at_20": 0.60,
    "baseline_precision_at_50": 0.66,
    "baseline_precision_at_100": 0.70,
    "baseline_precision_at_500": 0.666,

    "logistic_roc_auc": 0.588,
    "logistic_precision_at_20": 0.55,
    "logistic_precision_at_50": 0.54,
    "logistic_precision_at_100": 0.55,
    "logistic_precision_at_500": 0.504,

    "queue_rows": len(queue_export),
}
metrics_path = outputs_dir / "w07_playbook_metrics.json"

with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

print("Saved:", metrics_path)

Saved: ../outputs/w07_playbook_metrics.json


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.